# 1. Setup

Run the published 3Cv1 two-head checkpoint once on the official unseen images. This post-hoc diagnostic explains the existing result. Keep its outputs out of calibration, reranking and model selection.

Download the [released checkpoint](https://github.com/COS30082-ML-Project-2026/COS30082-Plant-Disease-OOD/releases/tag/approach3c_twohead) and set CHECKPOINT_PATH. The published run record is expected under results; update RECORD_PATH if it is elsewhere.

In [ ]:
import json, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_final_test_loaders
from common.evaluate import collect_scores, top_k_accuracy

SOURCE_MODEL = 'approach3c_dino_pairwise_feature_fusion_two_head'
MODEL_NAME = 'approach3c_two_head_unseen_top5_diagnostic'
CHECKPOINT_PATH = cfg.CHECKPOINT_DIR / f'{SOURCE_MODEL}.pt'
RUN_ID = SOURCE_MODEL + '_20261127_162656_659433'
RECORD_PATH = cfg.RESULTS_DIR / RUN_ID / f'{SOURCE_MODEL}_comparison.json'
cfg.NUM_WORKERS = 0
cfg.set_seed(cfg.SEED)

# 2. Data

Read only training-derived candidate rows. Use the published run metadata to verify the checkpoint and shared split before official access.

In [ ]:
pairs = get_pairs()

# 3. Model

Instantiate the original two-head architecture and load its full state dict. The scorer sums crop and disease logits for every candidate in common order.

In [ ]:
class Dino(nn.Module):
    def __init__(self, hp):
        super().__init__()
        self.crop = timm.create_model(hp['backbone'], pretrained=False,
                                      num_classes=cfg.NUM_CROPS, img_size=cfg.IMAGE_SIZE)
        self.disease = timm.create_model(hp['backbone'], pretrained=False,
                                         num_classes=cfg.NUM_DISEASES, img_size=cfg.IMAGE_SIZE)
        self.width = self.crop.num_features

    def forward(self, pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels), pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels), pre_logits=True).float())


def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


RECIPE = dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum', final_heads='crop_and_disease')

class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)
        self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
        self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
        self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
        self.fused_crop_head = nn.Linear(dim, cfg.NUM_CROPS)
        self.fused_disease_head = nn.Linear(dim, cfg.NUM_DISEASES)

    def features(self, images):
        crop, disease = self.backbone(images)
        return self.crop_projection(crop), self.disease_projection(disease)

    def score(self, crop, disease, candidate_pairs):
        combined = crop + disease
        tokens = torch.stack((crop + combined, disease + combined), dim=1)
        attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
        residuals = torch.stack((disease, crop), dim=1)
        image = self.image_fusion((attended + residuals).flatten(1))
        crop_ids, disease_ids = candidate_pairs.unbind(1)
        crop_scores = self.fused_crop_head(image)
        disease_scores = self.fused_disease_head(image)
        return crop_scores[:, crop_ids] + disease_scores[:, disease_ids]



def diagnose(scores, labels, final_pairs, development_pairs):
    seen_candidates = (final_pairs[:, None, :] == development_pairs[None, :, :]).all(dim=2).any(dim=1)
    if seen_candidates[labels].any():
        raise ValueError('The unseen loader contains a training-seen target.')
    # Use the two independent topk calls from common.evaluate, including its tie handling.
    top1 = scores.topk(1, dim=1).indices[:, 0]
    top5 = scores.topk(5, dim=1).indices
    hit1 = top1.eq(labels)
    hit5 = top5.eq(labels[:, None]).any(dim=1)
    near_miss = hit5 & ~hit1
    counts = seen_candidates[top5].sum(dim=1)
    top1_seen = seen_candidates[top1]
    slots = pd.DataFrame({'seen_slots': range(6), 'unseen_slots': range(5, -1, -1),
                          'images': torch.bincount(counts, minlength=6).tolist()})
    slots['pct_images'] = 100 * slots.images / len(labels)
    blockers = pd.DataFrame({
        'rank1_type': ['training_seen', 'other_unseen'],
        'images': [int((near_miss & top1_seen).sum()), int((near_miss & ~top1_seen).sum())],
    })
    blockers['pct_near_misses'] = (100 * blockers.images / int(near_miss.sum())
                                  if near_miss.any() else None)
    rows = pd.DataFrame(top5.numpy(), columns=[f'top5_position_{i}_pair_row' for i in range(1, 6)])
    rows = rows.assign(true_pair_row=labels.numpy(), top1_pair_row=top1.numpy(),
                       top1_seen=top1_seen.numpy(), seen_slots=counts.numpy(),
                       top1_correct=hit1.numpy(), top5_contains_target=hit5.numpy())
    return slots, blockers, rows

# 4. Prepare

Load the released checkpoint and check its identity, candidates, seed, epoch and shared split against the published record. This notebook has no training or tuning step.

In [ ]:
prepared = False
record = json.loads(Path(RECORD_PATH).read_text(encoding='utf8'))
selection = record['selection']
published = next(result for result in record['results'] if result['model'] == SOURCE_MODEL)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
fields = ('family', 'hp', 'recipe', 'best_epoch', 'best_val')
if (any(item.get('model_name') != SOURCE_MODEL or item.get('debug') is not False
        or item.get('depth') != 4 or item.get('recipe') != RECIPE for item in (selection, checkpoint))
        or any(selection[key] != checkpoint[key] for key in fields)
        or not torch.equal(checkpoint['pairs'], pairs) or selection['development_pairs'] != pairs.tolist()):
    raise ValueError('Expected the released four-block 3Cv1 two-head checkpoint and run record.')
HP = checkpoint['hp']
if (HP['backbone'], HP['norm'], HP['seed']) != ('vit_base_patch14_dinov2.lvd142m', 'imagenet', cfg.SEED):
    raise ValueError('Unexpected shared settings.')
train, val, _, _ = get_loaders(HP['batch_size'], norm=HP['norm'], debug=False)
if (selection['train_images'] != train.dataset.frame.image.tolist()
        or selection['validation_images'] != val.dataset.frame.image.tolist()):
    raise ValueError('Training or validation membership changed.')
model = Model(Dino(HP))
model.load_state_dict(checkpoint['state_dict'], strict=True)
model.to(cfg.DEVICE).eval().requires_grad_(False)
del checkpoint
print('Checkpoint loaded. Enable final diagnostic when ready.')
prepared = True

# 5. Evaluation

Explicitly opt in after checkpoint preparation. Compute standard Top-1 and Top-5 and their difference, with a count of ties where the separate Top-1 and Top-5 calculations disagree on membership. Break down near misses by the rank-1 prediction's training membership.

In [ ]:
RUN_FINAL_DIAGNOSTIC = False
summary = None
if RUN_FINAL_DIAGNOSTIC:
    if not globals().get('prepared', False):
        raise ValueError("Prepare and verify the checkpoint first.")
    final_pairs = get_pairs(final_evaluation=True)
    _, unseen = get_final_test_loaders(HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    candidates = final_pairs.to(cfg.DEVICE)

    @torch.no_grad()
    def predict(images):
        return model.score(*model.features(images.to(cfg.DEVICE)), candidates)

    scores, labels = collect_scores(predict, unseen)
    if scores.shape != (len(unseen.dataset), cfg.NUM_PAIRS):
        raise ValueError('Unexpected final score dimensions.')
    slots, blockers, per_image = diagnose(scores, labels, final_pairs, pairs)
    per_image.insert(0, 'image', unseen.dataset.frame.image.tolist())
    top1 = top_k_accuracy(scores, labels, 1)
    top5 = top_k_accuracy(scores, labels, 5)
    near_misses = int(blockers.images.sum())
    tie_disagreements = int((per_image.top1_correct & ~per_image.top5_contains_target).sum())
    summary = dict(n_images=len(labels), candidates=scores.shape[1], unseen_top1=top1,
                   unseen_top5=top5, near_misses=near_misses,
                   top1_correct_outside_top5=tie_disagreements,
                   published_unseen_top1=published['unseen_top1'],
                   published_unseen_top5=published['unseen_top5'],
                   reproduces_published=(top1, top5) == (
                       published['unseen_top1'], published['unseen_top5']))
    print(json.dumps(summary, indent=2))
    display(slots, blockers)

# 6. Save

Save the aggregate breakdown and per-image candidate row IDs under a new results folder.

In [ ]:
if summary is not None:
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    destination = cfg.RESULTS_DIR / f'{MODEL_NAME}_{stamp}'
    destination.mkdir(parents=True, exist_ok=False)
    (destination / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf8')
    slots.to_csv(destination / 'top5_seen_slot_distribution.csv', index=False)
    blockers.to_csv(destination / 'rank1_blockers.csv', index=False)
    per_image.to_csv(destination / 'per_image_top5.csv', index=False)
    print('Results:', destination)